# Danish Agricultural Subsidies - Exploratory Analysis

This notebook explores the silver subsidies datasets to:
1. Profile data quality (nulls, duplicates, formats)
2. Test validation hypotheses (payment × rate vs area)
3. Identify edge cases and anomalies
4. Document findings before building the gold pipeline

## Data Sources
- **støtteoplysninger.naturerhverv.dk** - EU payment data (DKK amounts)
- **Landbrugsstøtte_2023** - Field-level applications (hectares)
- **fvm_grassland_subsidies** - Spatial grassland subsidies
- **fvm_organic_subsidies** - Spatial organic subsidies
- **fvm_environmental_subsidies** - Spatial environmental subsidies

In [ ]:
import io

import numpy as np
import pandas as pd
from google.cloud import storage

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load Data from GCS

In [ ]:
# GCS paths for silver data
GCS_BUCKET = "landbruget-data"
SILVER_PATHS = {
    "stoetteoplysninger": (
        "silver/subsidies/20260110_192737/stoetteoplysninger.naturerhverv.dk_20241223_pii_handled.parquet"
    ),
    "landbrugsstoette": "silver/subsidies/20260110_192737/Landbrugsstoette_2023.parquet",
    "grassland": "silver/fvm_grassland_subsidies_2023/20260110_185622/data.parquet",
    "organic": "silver/fvm_organic_subsidies_2023/20260110_185512/data.parquet",
    "environmental": "silver/fvm_environmental_subsidies_2023/20260110_185742/data.parquet",
}


def load_parquet_from_gcs(bucket_name: str, blob_path: str) -> pd.DataFrame:
    """Load a parquet file from GCS into a pandas DataFrame."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    content = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(content))


# Load all datasets
print("Loading datasets from GCS...")
datasets = {}
for name, path in SILVER_PATHS.items():
    print(f"  Loading {name}...")
    datasets[name] = load_parquet_from_gcs(GCS_BUCKET, path)
    print(f"    Shape: {datasets[name].shape}")
print("Done!")

## 2. Data Profiling

In [ ]:
def profile_dataset(df: pd.DataFrame, name: str):
    """Generate a data profile for a DataFrame."""
    print(f"\n{'=' * 80}")
    print(f"PROFILE: {name}")
    print(f"{'=' * 80}")
    print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")

    print("\nColumn Types:")
    print(df.dtypes.to_string())

    print("\nNull Counts:")
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(1)
    null_df = pd.DataFrame({"nulls": null_counts, "pct": null_pct})
    print(null_df[null_df["nulls"] > 0].to_string() if null_df["nulls"].sum() > 0 else "  No nulls")

    # Check for CVR column
    cvr_cols = [c for c in df.columns if "cvr" in c.lower() or "vat" in c.lower()]
    if cvr_cols:
        cvr_col = cvr_cols[0]
        print(f"\nCVR Analysis ({cvr_col}):")
        print(f"  Unique values: {df[cvr_col].nunique():,}")
        # Check format
        sample = df[cvr_col].dropna().astype(str).head(5).tolist()
        print(f"  Sample values: {sample}")

    return null_df


# Profile all datasets
for name, df in datasets.items():
    profile_dataset(df, name)

## 3. støtteoplysninger Analysis - Double Counting Issue

In [ ]:
stoette = datasets["stoetteoplysninger"].copy()

# Convert amount columns to numeric
amount_cols = [c for c in stoette.columns if "dkk" in c.lower()]
for col in amount_cols:
    stoette[col] = pd.to_numeric(stoette[col], errors="coerce").fillna(0)

print("Amount columns:", amount_cols)
print("\nMeasure type distribution:")
print(stoette["measure_type_of_intervention"].value_counts())

In [ ]:
# Flag summary rows
stoette["is_summary_row"] = stoette["measure_type_of_intervention"] == "Total for beneficiary"

# Calculate totals
summary_rows = stoette[stoette["is_summary_row"]]
detail_rows = stoette[~stoette["is_summary_row"]]

print("DOUBLE COUNTING ANALYSIS")
print("=" * 60)
print(f"\nTotal rows: {len(stoette):,}")
print(f"Detail rows: {len(detail_rows):,}")
print(f"Summary rows ('Total for beneficiary'): {len(summary_rows):,}")

# Sum comparison
detail_eagf = detail_rows["amount_by_operation_under_eagf_dkk"].sum()
detail_eafrd = detail_rows["amount_by_operation_under_eafrd_dkk"].sum()
detail_total = detail_eagf + detail_eafrd

summary_total = summary_rows["total_of_the_eu_amount_for_that_beneficiary_dkk"].sum()

print(f"\nDetail rows - EAGF sum: {detail_eagf:,.0f} DKK")
print(f"Detail rows - EAFRD sum: {detail_eafrd:,.0f} DKK")
print(f"Detail rows - Total: {detail_total:,.0f} DKK")
print(f"\nSummary rows - Total: {summary_total:,.0f} DKK")

# Calculate difference percentage
diff = abs(detail_total - summary_total)
diff_pct = (diff / summary_total) * 100
print(f"\nDifference: {diff:,.0f} DKK ({diff_pct:.2f}%)")

print(f"\n⚠️ If summed together (WRONG): {detail_total + summary_total:,.0f} DKK (2x overcounting!)")

## 4. Known Subsidy Rates (2023)

Reference rates for validation (from Landbrugsstyrelsen):

In [ ]:
# 2023 Subsidy rates (kr/ha)
RATES_2023 = {
    "grundbetaling": 1999,  # Basic payment
    "biodiversitet_baeredygtighed": 2740,  # Bio-scheme
    "miljo_klimavenligt_graes": 1500,  # Environmental grass
    "varieret_planteproduktion": 615,  # Varied crops
    "ekstensivering_slaet": 3526,  # Extensification with mowing
    # Organic
    "okologisk_basis": 955,
    "okologisk_n_reduktion": 716,
    "okologisk_omlaegning": 1760,
    "okologisk_frugt_baer": 4404,
    # Grassland management (with grundbetaling)
    "pleje_afgraesning_med_grund": 1650,
    "pleje_slaet_med_grund": 850,
    # Grassland management (without grundbetaling)
    "pleje_afgraesning_uden_grund": 2600,
    "pleje_slaet_uden_grund": 1050,
}

print("Reference rates for 2023 (kr/ha):")
for scheme, rate in RATES_2023.items():
    print(f"  {scheme}: {rate:,} kr/ha")

## 5. Organic Subsidies Validation

Test hypothesis: `organic_payment ≈ organic_area × 955 kr/ha`

In [ ]:
# Load organic spatial data
organic = datasets["organic"].copy()
print(f"Organic subsidies shape: {organic.shape}")
print(f"\nColumns: {list(organic.columns)}")
print("\nSubsidy measures:")
print(organic["subsidy_measure"].value_counts())

In [ ]:
# Normalize CVR
organic["cvr"] = organic["cvr_number"].astype(str).str.zfill(8)

# Aggregate area by CVR
organic_by_cvr = (
    organic.groupby("cvr").agg({"area_ha": "sum", "field_id": "count"}).rename(columns={"field_id": "field_count"})
)

print(f"\nOrganic farms (unique CVRs): {len(organic_by_cvr):,}")
print(f"Total organic area: {organic_by_cvr['area_ha'].sum():,.2f} ha")
print(f"\nExpected payment at 955 kr/ha: {organic_by_cvr['area_ha'].sum() * 955:,.0f} DKK")

In [ ]:
# Get organic payments from støtteoplysninger
organic_payments = detail_rows[
    detail_rows["measure_type_of_intervention"].str.contains("Organic", case=False, na=False)
].copy()
organic_payments["cvr"] = organic_payments["vat_or_tax_identification_number"].astype(str).str.zfill(8)

print(f"\nOrganic payment rows in støtteoplysninger: {len(organic_payments):,}")
print(f"Unique CVRs with organic payments: {organic_payments['cvr'].nunique():,}")

# Sum payments by CVR
organic_payments_by_cvr = (
    organic_payments.groupby("cvr")
    .agg({"total_of_the_eu_amount_for_that_beneficiary_dkk": "sum"})
    .rename(columns={"total_of_the_eu_amount_for_that_beneficiary_dkk": "payment_dkk"})
)

print(f"\nTotal organic payments: {organic_payments_by_cvr['payment_dkk'].sum():,.0f} DKK")

In [ ]:
# Join area and payment data
organic_validation = organic_by_cvr.merge(
    organic_payments_by_cvr, left_index=True, right_index=True, how="outer"
).fillna(0)

# Calculate expected payment and variance
organic_validation["expected_payment"] = organic_validation["area_ha"] * RATES_2023["okologisk_basis"]
organic_validation["variance"] = organic_validation["payment_dkk"] - organic_validation["expected_payment"]
organic_validation["variance_pct"] = np.where(
    organic_validation["expected_payment"] > 0,
    organic_validation["variance"] / organic_validation["expected_payment"] * 100,
    np.nan,
)

print("\nORGANIC SUBSIDIES VALIDATION")
print("=" * 60)
print(f"CVRs with area data: {(organic_validation['area_ha'] > 0).sum():,}")
print(f"CVRs with payment data: {(organic_validation['payment_dkk'] > 0).sum():,}")
print(f"CVRs with both: {((organic_validation['area_ha'] > 0) & (organic_validation['payment_dkk'] > 0)).sum():,}")

print("\nVariance Statistics (for CVRs with both):")
both_mask = (organic_validation["area_ha"] > 0) & (organic_validation["payment_dkk"] > 0)
print(
    organic_validation.loc[
        both_mask,
        ["area_ha", "payment_dkk", "expected_payment", "variance", "variance_pct"],
    ].describe()
)

## 6. Grassland Subsidies Validation

In [ ]:
# Load grassland spatial data
grassland = datasets["grassland"].copy()
grassland["cvr"] = grassland["cvr_number"].astype(str).str.zfill(8)

print(f"Grassland subsidies shape: {grassland.shape}")
print("\nSubsidy measures:")
print(grassland["subsidy_measure"].value_counts())

In [ ]:
# Check subsidy type codes to understand with/without grundbetaling
print("\nSubsidy type codes:")
print(grassland["subsidy_type_code"].value_counts())

In [ ]:
# Aggregate grassland area by CVR
grassland_by_cvr = (
    grassland.groupby("cvr").agg({"area_ha": "sum", "field_id": "count"}).rename(columns={"field_id": "field_count"})
)

print(f"\nGrassland farms (unique CVRs): {len(grassland_by_cvr):,}")
print(f"Total grassland area: {grassland_by_cvr['area_ha'].sum():,.2f} ha")

# Expected payments (using average of with/without grundbetaling rates)
avg_rate = (RATES_2023["pleje_afgraesning_med_grund"] + RATES_2023["pleje_afgraesning_uden_grund"]) / 2
print("\nExpected payment range:")
print(f"  At 1,650 kr/ha (with grund): {grassland_by_cvr['area_ha'].sum() * 1650:,.0f} DKK")
print(f"  At 2,600 kr/ha (without grund): {grassland_by_cvr['area_ha'].sum() * 2600:,.0f} DKK")

## 7. Cross-Dataset CVR Overlap Analysis

In [ ]:
# Normalize CVRs across all datasets
stoette_cvrs = set(detail_rows["vat_or_tax_identification_number"].astype(str).str.zfill(8))
organic_cvrs = set(organic["cvr"])
grassland_cvrs = set(grassland["cvr"])

print("CVR OVERLAP ANALYSIS")
print("=" * 60)
print("\nUnique CVRs by dataset:")
print(f"  støtteoplysninger (payments): {len(stoette_cvrs):,}")
print(f"  Organic (spatial): {len(organic_cvrs):,}")
print(f"  Grassland (spatial): {len(grassland_cvrs):,}")

print("\nOverlaps:")
print(f"  støtteoplysninger ∩ Organic: {len(stoette_cvrs & organic_cvrs):,}")
print(f"  støtteoplysninger ∩ Grassland: {len(stoette_cvrs & grassland_cvrs):,}")
print(f"  Organic ∩ Grassland: {len(organic_cvrs & grassland_cvrs):,}")
print(f"  All three: {len(stoette_cvrs & organic_cvrs & grassland_cvrs):,}")

print("\nMissing from støtteoplysninger:")
print(f"  Organic CVRs not in payments: {len(organic_cvrs - stoette_cvrs):,}")
print(f"  Grassland CVRs not in payments: {len(grassland_cvrs - stoette_cvrs):,}")

## 8. National Total Validation

Denmark has ~2.6M ha agricultural land. Expected grundbetaling at ~1,999 kr/ha = ~5.2B DKK

In [ ]:
# Check grundbetaling from støtteoplysninger
grundbetaling = detail_rows[
    detail_rows["measure_type_of_intervention"].str.contains("Basic payment", case=False, na=False)
]

print("GRUNDBETALING (Basic Payment) VALIDATION")
print("=" * 60)
print(f"\nRows matching 'Basic payment': {len(grundbetaling):,}")
print(f"Unique CVRs: {grundbetaling['vat_or_tax_identification_number'].nunique():,}")
print(f"\nTotal grundbetaling paid: {grundbetaling['amount_by_operation_under_eagf_dkk'].sum():,.0f} DKK")

# Estimate area from payment
estimated_area = grundbetaling["amount_by_operation_under_eagf_dkk"].sum() / RATES_2023["grundbetaling"]
print(f"\nEstimated area (payment / 1,999 kr/ha): {estimated_area:,.0f} ha")
print("Expected (Denmark ~2.6M ha): ~2,600,000 ha")
print(f"Coverage: {estimated_area / 2600000 * 100:.1f}%")

## 9. Summary & Findings

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY OF FINDINGS")
print("=" * 80)

print("""
1. DOUBLE COUNTING ISSUE CONFIRMED
   - støtteoplysninger contains both detail rows AND 'Total for beneficiary' summary rows
   - Summing all rows would result in ~2x overcounting
   - SOLUTION: Add is_summary_row flag and exclude summaries from aggregations

2. DATA OVERLAP
   - Spatial datasets (organic, grassland) overlap with payment data
   - Not all spatial CVRs appear in payment data (timing differences?)
   - Cross-validation possible using rates x area

3. RATE VALIDATION
   - Payment/area ratios can be compared against known 2023 rates
   - Variances may indicate:
     * Multiple subsidy schemes stacking
     * Different rate tiers (with/without grundbetaling)
     * Data quality issues

4. RECOMMENDATIONS FOR GOLD LAYER
   - Keep detail and summary rows separate (is_summary_row flag)
   - Normalize all CVRs to 8-digit strings
   - Add expected_payment calculated field for validation
   - Document all known overlaps in metadata
""")